# Feature Engineering — Guided Examples

This notebook uses synthetic customer data to demonstrate common feature-engineering methods. It is designed to run locally without downloading a dataset.

**Workflow:** inspect raw fields → create domain features → split → build preprocessing pipeline → evaluate → inspect feature selection and leakage risks.

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
from sklearn.datasets import make_classification

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print("Ready. Random state:", RANDOM_STATE)

## 2. Create a synthetic dataset

In [ ]:
n = 1200
raw = pd.DataFrame({
    "age": rng.integers(18, 80, size=n).astype(float),
    "monthly_spend": rng.gamma(shape=4, scale=30, size=n),
    "visits_30d": rng.poisson(lam=5, size=n),
    "support_tickets": rng.poisson(lam=1.2, size=n),
    "region": rng.choice(["North", "South", "East", "West"], size=n),
    "plan": rng.choice(["Basic", "Plus", "Premium"], p=[0.5, 0.32, 0.18], size=n),
    "signup_date": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, size=n), unit="D"),
})
# Construct a synthetic outcome with a relationship to a few raw fields.
logit = (
    -1.5
    - 0.12 * raw["visits_30d"]
    + 0.30 * raw["support_tickets"]
    + 0.002 * raw["monthly_spend"]
    + 0.25 * (raw["plan"] == "Basic").astype(int)
)
prob = 1 / (1 + np.exp(-logit))
raw["churn"] = rng.binomial(1, prob)
# Add missing values to illustrate pipeline imputation.
for col in ["age", "monthly_spend", "region"]:
    ix = rng.choice(raw.index, size=30, replace=False)
    raw.loc[ix, col] = np.nan

raw.head()

## 3. Inspect data quality

In [ ]:
print("Shape:", raw.shape)
display(raw.dtypes.to_frame("dtype"))
display(raw.isna().sum().to_frame("missing values"))
print("Duplicate rows:", raw.duplicated().sum())
display(raw["churn"].value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
raw[["age", "monthly_spend", "visits_30d", "support_tickets"]].hist(
    figsize=(10, 6), bins=25
)
plt.suptitle("Raw numeric feature distributions")
plt.tight_layout()
plt.show()

## 4. Create meaningful features
Features should have a clear definition and be available at prediction time. These examples create activity and account-age features.

In [ ]:
df = raw.copy()
df["tickets_per_visit"] = np.where(
    df["visits_30d"] > 0,
    df["support_tickets"] / df["visits_30d"],
    np.nan
)
df["spend_per_visit"] = np.where(
    df["visits_30d"] > 0,
    df["monthly_spend"] / df["visits_30d"],
    np.nan
)
reference_date = pd.Timestamp("2025-06-30")
df["account_age_days"] = (reference_date - df["signup_date"]).dt.days
df["signup_month"] = df["signup_date"].dt.month
df["signup_day_of_week"] = df["signup_date"].dt.dayofweek
df[["visits_30d", "support_tickets", "tickets_per_visit", "spend_per_visit",
    "account_age_days", "signup_month", "signup_day_of_week"]].head()

## 5. Inspect skew and compare transformations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
raw["monthly_spend"].dropna().hist(ax=axes[0], bins=30)
axes[0].set_title("Monthly spend")
np.log1p(raw["monthly_spend"].dropna()).hist(ax=axes[1], bins=30)
axes[1].set_title("log1p(monthly spend)")
plt.tight_layout()
plt.show()

In [ ]:
df["log_monthly_spend"] = np.log1p(df["monthly_spend"])
print("Original skew:", round(df["monthly_spend"].skew(), 3))
print("Log-transformed skew:", round(df["log_monthly_spend"].skew(), 3))

## 6. Outlier flags with IQR
The IQR rule is a screening heuristic, not proof that an observation is invalid. In a real pipeline, thresholds must be learned from training data.

In [ ]:
col = df["monthly_spend"]
q1, q3 = col.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["monthly_spend_iqr_flag"] = ~col.between(lower, upper)
print("IQR bounds:", round(lower, 2), round(upper, 2))
print("Flagged rows:", int(df["monthly_spend_iqr_flag"].sum()))

## 7. Split before fitting learned transformations
Drop the raw date after deriving allowed features. The synthetic target is generated from the available fields, and all features below are illustrative.

In [ ]:
target = "churn"
drop_cols = [target, "signup_date", "monthly_spend_iqr_flag"]  # exploratory full-data flag is not used as a model feature
X = df.drop(columns=drop_cols)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

## 8. Build a preprocessing and modeling pipeline
Numeric columns are median-imputed and standardized. Categorical columns are imputed and one-hot encoded. Learned transformations are fitted only on training folds.

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
print("Pipeline fitted.")

## 9. Compare a baseline and the engineered model

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print("Baseline")
print(classification_report(y_test, base_pred, zero_division=0))
print("Feature-engineered pipeline")
print(classification_report(y_test, pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Feature-engineered pipeline — test set")
plt.show()

## 10. Feature selection inside a pipeline
Feature selection must be fitted within the training process. Here, SelectKBest uses an ANOVA F-test and selects up to 10 features after preprocessing.

In [ ]:
selection_model = Pipeline([
    ("preprocess", preprocess),
    ("select", SelectKBest(score_func=f_classif, k=10)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
selection_model.fit(X_train, y_train)
selected_pred = selection_model.predict(X_test)
selected_proba = selection_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, selected_pred, zero_division=0))
print("Selected pipeline ROC-AUC:", round(roc_auc_score(y_test, selected_proba), 3))

## 11. Cross-validation on the training split

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)
pd.DataFrame({
    metric.replace("test_", ""): [cv_scores[metric].mean(), cv_scores[metric].std()]
    for metric in cv_scores if metric.startswith("test_")
}, index=["mean", "std"]).T

## 12. Demonstrate scaling behavior

In [ ]:
sample = np.array([[1.0], [2.0], [3.0], [100.0]])
scaled = pd.DataFrame({
    "raw": sample.ravel(),
    "standard": StandardScaler().fit_transform(sample).ravel(),
    "minmax": MinMaxScaler().fit_transform(sample).ravel(),
    "robust": RobustScaler().fit_transform(sample).ravel(),
})
scaled

## 13. Feature specification and leakage review

In [ ]:
feature_spec = pd.DataFrame([
    ["tickets_per_visit", "Support tickets per recent visit", "support_tickets, visits_30d",
     "support_tickets / visits_30d; missing when visits=0", "Numeric", "Known at scoring time? Verify"],
    ["spend_per_visit", "Spend per recent visit", "monthly_spend, visits_30d",
     "monthly_spend / visits_30d; missing when visits=0", "Numeric", "Known at scoring time? Verify"],
    ["account_age_days", "Days since signup", "signup_date, reference timestamp",
     "reference timestamp - signup_date", "Numeric", "Use the actual scoring timestamp"],
], columns=["feature", "meaning", "source_fields", "formula", "type", "availability_check"])
feature_spec

## 14. Takeaways and extension tasks

- Add at least one feature and state the hypothesis it represents.
- Compare with and without that feature using the same cross-validation folds.
- Try `RobustScaler` or a different imputation strategy.
- Inspect whether the feature is available at the exact prediction time.
- Do not use the held-out test set to repeatedly choose features.

**Important:** this notebook is educational. Synthetic patterns are not evidence about real customers, and good offline metrics alone are not sufficient for deployment.